# Revenue Bridge Analysis
### Strategic Risk Framework by Mohamed Bah

This notebook executes the full data pipeline: generating mock SaaS datasets, performing a 3-way join, and calculating **Revenue at Risk**.

In [ ]:
# 1. Setup and Data Generation
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
customers = [f"Company {i}" for i in range(1, 101)]
crm_data = pd.DataFrame({
    'customer_name': customers,
    'tier': [random.choice(['Enterprise', 'Mid-Market', 'SMB']) for _ in range(100)],
    'mrr': [random.randint(500, 15000) for _ in range(100)]
})

support_data = pd.DataFrame({
    'customer_name': [random.choice(customers) for _ in range(200)],
    'sentiment_score': [round(random.uniform(1, 5), 1) for _ in range(200)]
})

usage_data = pd.DataFrame({
    'customer_name': customers,
    'last_login_days_ago': [random.randint(0, 45) for _ in range(100)],
    'feature_adoption_rate': [round(random.uniform(0.1, 0.9), 2) for _ in range(100)]
})
print("✅ Datasets generated in memory.")

In [ ]:
# 2. Perform Join and Calculate Health Score
support_agg = support_data.groupby('customer_name').agg({'sentiment_score': 'mean'}).reset_index()
df = pd.merge(crm_data, usage_data, on='customer_name')
df = pd.merge(df, support_agg, on='customer_name', how='left').fillna(3.0)

def calculate_health(row):
    engagement = max(0, 100 - (row['last_login_days_ago'] * 3))
    sentiment = row['sentiment_score'] * 20
    adoption = row['feature_adoption_rate'] * 100
    return (engagement * 0.4) + (sentiment * 0.3) + (adoption * 0.3)

df['health_score'] = df.apply(calculate_health, axis=1)
at_risk = df[(df['mrr'] > 5000) & (df['health_score'] < 50)].sort_values(by='mrr', ascending=False)

print(f"--- REVENUE RISK REPORT ---")
print(f"Accounts at Risk: {len(at_risk)}")
print(f"Total MRR at Risk: ${at_risk['mrr'].sum():,.2f}")
at_risk.head()